# MNAI Nano 2 — 0.9B from scratch — Colab, step by step

**Do every cell in order, top to bottom. Do not skip one.**

Made by MeowSamOPDZN Studio. Owner: MeowSamOP DZN. Full form: Meow x Nythera AI.

What this notebook does:
1. Checks you have a GPU
2. Clones the public repo
3. Installs everything
4. Trains your own tokenizer
5. Trains the 0.9B model (with resume, so a disconnect does not lose it)
6. Tests it
7. Turns it into a GGUF you can download and run in LM Studio

**Before you start:** in Colab, go to Runtime > Change runtime type > Hardware accelerator > T4 GPU > Save. If it says CPU, stop, change it first.

## Step 1 — Check the GPU

Run this. You should see a Tesla T4, 15GB or 16GB. If you see no GPU, change the runtime type first (see top).

In [ ]:
!nvidia-smi

## Step 2 — Clone the repo

This clones the public main repo. It is public, so no login or token is needed.

In [ ]:
!git clone https://github.com/MeowSamOPDZN/mnai-nano-2.git
%cd mnai-nano-2
!ls -lh

## Step 3 — Install everything

This installs torch, transformers, datasets, tokenizers and bitsandbytes. It takes a few minutes the first time.

In [ ]:
!pip install -q -r requirements.txt
!python -c "import torch; print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())"

## Step 4 — Check the model shape

This proves the 0.9B model loads and runs. It should print 899,753,472 params (0.900B) and a logits shape ending in 32768. If this fails, do not go on to training, tell me the error.

In [ ]:
!python model.py

## Step 5 — Save your work to Google Drive

**Do this before training.** Colab deletes everything when the session ends. This mounts Drive and makes training save there, so you can resume later.

Run it, click the link, pick your Google account, allow it, copy the code back into the box.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mnai-nano-2
!ln -sfn /content/drive/MyDrive/mnai-nano-2 /content/mnai-nano-2/out-09b
!ls /content/drive/MyDrive/mnai-nano-2

## Step 6 — Train the tokenizer (once)

This trains your own 32768-token BPE tokenizer from FineWeb-Edu. Run it once, it takes a few minutes. It saves tokenizer.json in this folder. Do not train the model before this is done.

In [ ]:
!python train_tokenizer.py --out tokenizer.json --num-docs 50000
!ls -lh tokenizer.json

## Step 7 — Train the model (first run)

This is the real training. 1B tokens is the first useful target. **It will NOT finish in one Colab session.** Colab stops after about 12 hours, sometimes sooner. That is normal.

Just let it run as long as you can, then go to Step 8 to resume. Checkpoints save every 500 steps to out-09b, which is on Drive from Step 5.

In [ ]:
!python train.py --tokenizer tokenizer.json --out out-09b --max-tokens 1000000000 --micro-batch 4 --grad-accum 16

## Step 8 — Resume training (after a disconnect)

Run this instead of Step 7 when you come back. First list the checkpoints, find the newest ckpt number, put that number in the resume path below, then run it.

Example: if the newest folder is ckpt-1500, use --resume out-09b/ckpt-1500

In [ ]:
!ls out-09b
!ls /content/drive/MyDrive/mnai-nano-2

Change ckpt-500 below to your newest checkpoint, then run this cell:

In [ ]:
!python train.py --tokenizer tokenizer.json --out out-09b --resume out-09b/ckpt-500 --max-tokens 1000000000 --micro-batch 4 --grad-accum 16

## Step 9 — Test the model

Change ckpt-500 below to your newest checkpoint, then run it. It should complete the sentence. Early checkpoints will sound like nonsense, that is normal, it gets better with more tokens.

In [ ]:
!python generate.py --model out-09b/ckpt-500 --prompt "MNAI was created by" --max-new 120

## Step 10 — Make the GGUF

Change ckpt-500 below to the checkpoint you want to use. This clones llama.cpp, converts the checkpoint to GGUF, then quantizes it to Q4_K_M. The file is about 550MB and runs on a 4GB RAM laptop in LM Studio.

In [ ]:
!python convert_to_gguf.py --model out-09b/ckpt-500 --out mnai-nano-2-q4_k_m.gguf
!ls -lh *.gguf

## Step 11 — Download the GGUF

Run this, then download the file from the file browser on the left (folder icon) — right-click mnai-nano-2-q4_k_m.gguf > Download. Load it in LM Studio.

In [ ]:
!ls -lh mnai-nano-2-q4_k_m.gguf
print('done — download mnai-nano-2-q4_k_m.gguf from the file browser on the left')

## If something goes wrong

- **No GPU in Step 1:** Runtime > Change runtime type > T4 GPU, then Runtime > Restart runtime, run from Step 1 again.
- **Out of memory in Step 7:** lower micro-batch to 2: add --micro-batch 2 --grad-accum 32 to the train command.
- **Session ended:** do Step 5 again to mount Drive, then Step 8 to resume. Do not start Step 7 from scratch.
- **tokenizer.json missing:** run Step 6 again, it must exist before Step 7.